# Gaussian Message Passing

Message passing algorithms are typically presented using finite, discrete probability distributions. The ideas generalize more broadly, but the key computational snag is that it becomes hard to compute/represent the messages.

<!-- TEASER_END -->

## Key operations

The general message passing algorithm requires two key operations: `update` and `marginalize`. The `update` function folds in information about observed data into a message. The `marginalize` function reduces the scope of a message to a smaller set of variables.

## Code

Let's start by defining some tools for implementing these ideas

In [1]:
from functools import partial

import numpy as np

from numpy.linalg import cholesky
from scipy.linalg import solve_triangular
from scipy.stats import multivariate_normal as mvn

solve_tril = partial(solve_triangular, lower=True)
solve_triu = partial(solve_triangular, lower=False)

def cho_solve(L, b):
    y = solve_tril(L, b)
    x = solve_triu(L.T, y)
    return x

def cho_inv(L):
    return cho_solve(L, np.eye(len(L)))

def cho_logdet(L):
    return np.sum(2.0 * np.log(np.diag(L)))

## Information form

The density of a multivariate normal given its mean and covariance is

$$
p(x \mid m, S) =
  \exp\left\{
    x^\top S^{-1} m
    -\frac{1}{2} x^\top S^{-1} x
  \right\}
  \exp\left\{
    -\frac{1}{2} m^\top S^{-1} m
    -\frac{n}{2} \log 2\pi
    -\frac{1}{2} \log |S|
  \right\}
$$

If we reparameterize the density using the definitions $h = S^{-1}m$ and $J = S^{-1}$, then we get the *information form* of the multivariate normal.

$$
p(x \mid h, J) =
  \exp\left\{
    x^\top h
    -\frac{1}{2} x^\top J x
  \right\}
  \exp\left\{
    -\frac{1}{2} h^\top J^{-1} h
    -\frac{n}{2} \log 2\pi
    +\frac{1}{2} \log |J|
  \right\}
$$

We can read off the normalization constants.

$$
\begin{align}
\log Z(m, S) &=
  \frac{1}{2} m^\top S^{-1} m
  + \frac{n}{2} \log 2\pi
  + \frac{1}{2} \log |S| \\
\log Z(h, J) &=
  \frac{1}{2} h^\top J^{-1} h
  + \frac{n}{2} \log 2\pi
  - \frac{1}{2} \log |J|
\end{align}
$$

This tells us that

$$
\int \exp\left\{
  x^\top h - \frac{1}{2} x^\top J x
\right\} {\sf d} x
=
Z(h, J),
$$
which we will use to derive our marginalization result. To do this, we write a distribution over vector $x$ as
$$
\begin{bmatrix} x_1 \\ x_2 \end{bmatrix}
\sim
\mathcal{N} \left(
\begin{bmatrix} h_1 \\ h_2 \end{bmatrix},
\begin{bmatrix}
  J_{11} & J_{12} \\
  J_{21} & J_{22}
\end{bmatrix}
\right)
$$

The distribution is
$$
\log p\left(
\begin{bmatrix} x_1 \\ x_2 \end{bmatrix}
\right)
=
x_1^\top h_1 + x_2^\top h_2
-\frac{1}{2} x_1^\top J_{11} x_1
-\frac{1}{2} x_2^\top J_{22} x_2
- x_1^\top J_{12} x_2
- \log Z(h, J)
$$
We can then compute
$$
\begin{align}
p(x_1) &=
  Z^{-1}(h, J)
  \exp\left\{
    x_1^\top h_1 - \frac{1}{2} x_1^\top J_{11} x_1
  \right\}
  \int \exp\left\{
    x_2^\top (h_2 - J_{21} x_1)
    -\frac{1}{2} x_2^\top J_{22} x_2
  \right\} {\sf d} x_2
  \\
  &\propto
  \exp\left\{
    x_1^\top h_1 - \frac{1}{2} x_1^\top J_{11} x_1
  \right\}
  \int \exp\left\{
    x_2^\top (h_2 - J_{21} x_1)
    -\frac{1}{2} x_2^\top J_{22} x_2
  \right\} {\sf d} x_2
  \\
  &\propto
  \exp\left\{
    x_1^\top h_1
    - \frac{1}{2} x_1^\top J_{11} x_1
    + \frac{1}{2} (h_2 - J_{21} x_1)^\top J_{22}^{-1} (h_2 - J_{21} x_1)
  \right\}
  \\
  &\propto
  \exp\left\{
    x_1^\top (h_1 - J_{12} J_{22}^{-1} h_2)
    - \frac{1}{2} x_1^\top (J_{11} - J_{12} J_{22}^{-1} J_{21}) x_1
  \right\}
\end{align}
$$

Here are some functions that we can use to switch back and forth between the two parameterizations, and compute the log probability of an observation for each form of the multivariate normal.

In [3]:
def moment_to_info(m, S):
    L = cholesky(S)
    h = cho_solve(L, m)
    J = cho_inv(L)
    return h, J

def info_to_moment(h, J):
    L = cholesky(J)
    m = cho_solve(L, h)
    S = cho_inv(L)
    return m, S

def mvn_logpdf_moment(x, m, S):
    L = cholesky(S)
    x = solve_tril(L, x)
    m = solve_tril(L, m)
    return (np.dot(x, m)
            - 0.5 * np.dot(x, x)
            - 0.5 * np.dot(m, m)
            - 0.5 * len(x) * np.log(2.0 * np.pi)
            - 0.5 * cho_logdet(L))

def mvn_logpdf_info(x, h, J):
    L = cholesky(J)
    g = solve_tril(L, h)
    return (np.dot(x, h)
            - 0.5 * np.dot(x, np.dot(J, x))
            - 0.5 * np.dot(g, g)
            - 0.5 * len(x) * np.log(2.0 * np.pi)
            + 0.5 * cho_logdet(L))

And we can test that these work.

In [5]:
m = np.random.normal(size=3)
A = np.random.normal(size=(3, 3))
S = np.dot(A.T, A)
L = cholesky(S)
h, J = moment_to_info(m, S)
x = np.random.normal(size=3)

mvn.logpdf(x, m, S), mvn_logpdf_moment(x, m, S), mvn_logpdf_info(x, h, J)

(-7.964807622655513, -7.964807622655533, -7.964807622655533)

The information form of the multivariate normal is 